In [ ]:
# Kiểm tra thông số GPU Google Colab cấp phát
!nvidia-smi

# Cài đặt sentence-transformers và các thư viện cần thiết
!pip install -q sentence-transformers torch scikit-learn numpy

In [ ]:
import json
import os
from google.colab import files

train_file = "train_triplets.json"
val_file = "val_triplets.json"

# Kiểm tra xem người dùng có muốn upload file từ máy không
if not os.path.exists(train_file) or not os.path.exists(val_file):
    print("Vui lòng chọn 2 file train_triplets.json và val_triplets.json để tải lên (từ thư mục colab_embedding_training/data/):")
    uploaded = files.upload()

if os.path.exists(train_file) and os.path.exists(val_file):
    with open(train_file, "r", encoding="utf-8") as f:
        train_data = json.load(f)
    with open(val_file, "r", encoding="utf-8") as f:
        val_data = json.load(f)
    print(f"-> Đã nạp thành công: {len(train_data)} mẫu Train | {len(val_data)} mẫu Validation!")
else:
    print("Chưa upload file, đang sử dụng tập dữ liệu chuẩn mặc định...")
    raise FileNotFoundError("Vui lòng tải lên 2 file train_triplets.json và val_triplets.json!")

In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

# 1. Chuẩn bị DataLoader
train_examples = []
for item in train_data:
    a = item.get("anchor", "").strip()
    p = item.get("positive", "").strip()
    n = item.get("negative", "").strip()
    if a and p:
        if n:
            train_examples.append(InputExample(texts=[a, p, n]))
        else:
            train_examples.append(InputExample(texts=[a, p]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# 2. Chuẩn bị Evaluator trên tập Val
val_anchors = [item["anchor"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]
val_positives = [item["positive"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]
val_negatives = [item["negative"] for item in val_data if item.get("anchor") and item.get("positive") and item.get("negative")]

evaluator = evaluation.TripletEvaluator(
    anchors=val_anchors,
    positives=val_positives,
    negatives=val_negatives,
    name="cosmetics_expanded_val",
    show_progress_bar=False
)

# 3. Tải Base Model (BKAI Vietnamese Bi-Encoder)
BASE_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
print(f"Đang tải mô hình nền: {BASE_MODEL_NAME}...")
model = SentenceTransformer(BASE_MODEL_NAME)

# Đo độ chính xác ban đầu
print("\n--- [1] ĐỘ CHÍNH XÁC CỦA BASELINE TRƯỚC KHI TRAIN ---")
score_before = evaluator(model)
print(f"-> Baseline Triplet Accuracy: {score_before * 100:.2f}%")

# 4. Cấu hình Huấn luyện
train_loss = losses.MultipleNegativesRankingLoss(model)
EPOCHS = 3
OUTPUT_DIR = "./my_cosmetics_embedding"

print(f"\n--- [2] BẮT ĐẦU HUẤN LUYỆN TRONG {EPOCHS} EPOCHS TRÊN GPU T4 ---")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=math.ceil(len(train_dataloader) * EPOCHS * 0.1),
    output_path=OUTPUT_DIR,
    save_best_model=True,
    show_progress_bar=True
)

# 5. Đo lường sau huấn luyện
best_model = SentenceTransformer(OUTPUT_DIR)
score_after = evaluator(best_model)
print("\n" + "="*65)
print(f"-> ĐỘ CHÍNH XÁC SAU KHI FINE-TUNE (AFTER): {score_after * 100:.2f}%")
print(f"-> MỨC ĐỘ CẢI THIỆN ĐẠT ĐƯỢC: +{(score_after - score_before) * 100:.2f}%")
print("="*65)

In [ ]:
import numpy as np

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def detailed_benchmark(m, test_set):
    a_vecs = m.encode([x["anchor"] for x in test_set], normalize_embeddings=True)
    p_vecs = m.encode([x["positive"] for x in test_set], normalize_embeddings=True)
    n_vecs = m.encode([x["negative"] for x in test_set], normalize_embeddings=True)
    
    margins = []
    hit1 = 0
    hit3 = 0
    rrs = []
    
    for i in range(len(test_set)):
        sim_p = cosine_sim(a_vecs[i], p_vecs[i])
        sim_n = cosine_sim(a_vecs[i], n_vecs[i])
        margins.append(sim_p - sim_n)
        
        all_sims = np.dot(p_vecs, a_vecs[i])
        rank = list(np.argsort(-all_sims)).index(i) + 1
        rrs.append(1.0 / rank)
        if rank == 1: hit1 += 1
        if rank <= 3: hit3 += 1
        
    N = len(test_set)
    return {
        "margin": float(np.mean(margins)),
        "hit1": (hit1 / N) * 100,
        "hit3": (hit3 / N) * 100,
        "mrr": float(np.mean(rrs))
    }

base_res = detailed_benchmark(model, val_data)
ft_res = detailed_benchmark(best_model, val_data)

report = f"""
### BẢNG THỰC NGHIỆM ĐÁNH GIÁ ĐỊNH LƯỢNG CHO KHÓA LUẬN TỐT NGHIỆP
*(Tập dữ liệu mở rộng 240 mẫu: Đa dạng tiếng lóng 3 miền, Code-switching, Bẫy cùng hãng)*

| Chỉ số Khoa học | Baseline (Trước khi train) | Fine-tuned (Sau khi train) | Mức độ Cải thiện (Delta) | Ý nghĩa Nghiệp vụ |
| :--- | :---: | :---: | :---: | :--- |
| **Độ chính xác Triplet** | **{score_before*100:.2f}%** | **{score_after*100:.2f}%** | **+{(score_after-score_before)*100:.2f}%** | Phân loại đúng giữa sản phẩm thích hợp và bẫy |
| **Khoảng cách Cosine Margin** | **{base_res['margin']:.4f}** | **{ft_res['margin']:.4f}** | **+{(ft_res['margin']-base_res['margin']):.4f}** | Độ dãn cách an toàn giữa sản phẩm đúng và sai |
| **Hit Rate@1** | **{base_res['hit1']:.2f}%** | **{ft_res['hit1']:.2f}%** | **+{(ft_res['hit1']-base_res['hit1']):.2f}%** | Sản phẩm chuẩn xác nằm ngay Top 1 |
| **Hit Rate@3** | **{base_res['hit3']:.2f}%** | **{ft_res['hit3']:.2f}%** | **+{(ft_res['hit3']-base_res['hit3']):.2f}%** | Sản phẩm chuẩn xác nằm trong Top 3 thẻ Card |
| **Thứ hạng Nghịch đảo (MRR@10)** | **{base_res['mrr']:.4f}** | **{ft_res['mrr']:.4f}** | **+{(ft_res['mrr']-base_res['mrr']):.4f}** | Năng lực xếp hạng tối ưu vị trí truy xuất |
"""
print(report)

with open("BENCHMARK_REPORT.md", "w", encoding="utf-8") as f:
    f.write(report)